In [1]:
import lattice, os, json
import pandas as pd
from dataclasses import dataclass, asdict, field
from typing import ClassVar

In [2]:
os.environ['DEMO_KEY'] = 'A66DPXZM'
os.environ['DEMO_SECRET'] = 'ivcmneyhhs7c4eg6'
os.environ['DEMO_SERVER'] = 'https://lattice-api-long.demo.lattice-data.org'
conn = lattice.Connection('demo')

In [170]:
test_lib = lattice.get_object('fd01a7a6-74fa-4a0a-876b-b964e776ecc8',conn,frame='object')

In [171]:
test

{'lab': '/labs/alex-marson/',
 'status': 'current',
 'aliases': ['alex-marson:CD4i_R1L08_CRI'],
 'samples': ['/tissues/b9f3df6d-e6a6-4663-b90e-16ca8f123b57/',
  '/tissues/bc0e96d2-ee51-437e-a837-e41fd233ba42/',
  '/tissues/8cc71ca7-e894-40ab-82c2-3376d57642d3/',
  '/tissues/12ee1bd4-9b24-42a9-98e2-f13431e2676b/'],
 'submitted_by': '/users/373c027a-53e8-45f3-852a-94ea5be43a69/',
 'schema_version': '1',
 'linked_libraries': ['/droplet_based_libraries/27a84742-c565-4bb9-a3b6-88472c1c23d3/'],
 'chemistry_version': 'Flex v1',
 'creation_timestamp': '2026-04-06T18:26:47.012531+00:00',
 'library_cardinality': 'dual',
 '@id': '/droplet_based_libraries/fd01a7a6-74fa-4a0a-876b-b964e776ecc8/',
 '@type': ['DropletBasedLibrary', 'Library', 'Item'],
 'uuid': 'fd01a7a6-74fa-4a0a-876b-b964e776ecc8',
 'summary': 'alex-marson:CD4i_R1L08_CRI'}

In [172]:
test_tissue = lattice.get_object('b9f3df6d-e6a6-4663-b90e-16ca8f123b57',conn,frame='object')
test_tissue

{'lab': '/labs/alex-marson/',
 'donors': ['/human_donors/19a3f946-0c86-40e1-bcb9-cdee76c41fa8/'],
 'status': 'current',
 'aliases': ['alex-marson:CD4i_R1L08_D1_Rest'],
 'age_units': 'year',
 'sample_terms': ['/controlled_terms/9c19f87e-8192-4de1-bfce-efa6205a8f44/'],
 'submitted_by': '/users/373c027a-53e8-45f3-852a-94ea5be43a69/',
 'schema_version': '1',
 'lower_bound_age': 28,
 'suspension_type': 'cell',
 'upper_bound_age': 29,
 'creation_timestamp': '2026-04-06T18:01:26.066658+00:00',
 'enriched_cell_types': ['/controlled_terms/ca46f4b7-9c2c-45df-8636-6d10945b8a44/'],
 'preservation_method': 'fixed-frozen',
 'genetic_modification': '/genetic_modifications/2007be6b-b88f-47b5-9393-405ca1fa8f52/',
 '@id': '/tissues/b9f3df6d-e6a6-4663-b90e-16ca8f123b57/',
 '@type': ['Tissue', 'Biosample', 'Item'],
 'uuid': 'b9f3df6d-e6a6-4663-b90e-16ca8f123b57',
 'summary': 'alex-marson:CD4i_R1L08_D1_Rest'}

In [173]:
check = test_tissue['preservation_method'] if 'preservation_method' in test_tissue else None
check

'fixed-frozen'

In [174]:
test_sample_term = lattice.get_object('9c19f87e-8192-4de1-bfce-efa6205a8f44',conn,frame='object')
test_sample_term

{'status': 'current',
 'term_id': 'UBERON:0000178',
 'term_name': 'blood',
 'submitted_by': '/users/373c027a-53e8-45f3-852a-94ea5be43a69/',
 'schema_version': '1',
 'ontology_source': 'UBERON',
 'creation_timestamp': '2026-04-03T23:28:44.638838+00:00',
 '@id': '/controlled_terms/9c19f87e-8192-4de1-bfce-efa6205a8f44/',
 '@type': ['ControlledTerm', 'Item'],
 'uuid': '9c19f87e-8192-4de1-bfce-efa6205a8f44',
 'summary': 'blood'}

In [175]:
# For now, assuming receiving a list of libraries uuids?
# Need a get_libraries function, which takes list of uuids and gathers
# connection would be a constant, for now just hard coded to conn
# Need to add handling for if the field is not present in the json object, e.x. Biosample.collection_date

In [10]:
library_uuids = ['fd01a7a6-74fa-4a0a-876b-b964e776ecc8','fa18ec20-cbc2-4b90-ba95-998e885f4336','f979fb3b-4c98-40d0-8731-7ebfde3d7cc8']

In [11]:
# Testing registry usage with donor, so that if a donor json object has already been pulled, it won't be pulled again
@dataclass
class Donor:
    """
    Dataclass to hold parsed information for donor
    """
    uuid: str
    ethnicity : str = field(init=False)
    sex : str = field(init=False)

    _instances: ClassVar[dict[str, "Donor"]] = {}  # Class-level registry

    def __post_init__(self):
        self.json_object : object = lattice.get_object(self.uuid,conn,frame='object')
        self.ethnicity : str = lattice.get_object((self.json_object['ethnicity'].split('/')[2]),conn,frame='object')['term_name']
        self.sex : str = self.json_object['sex']
    
    @classmethod
    def get_or_create(cls, uuid: str) -> "Donor":
        """Return existing Donor if UUID exists, otherwise create and register a new one."""
        if uuid not in cls._instances:
            cls._instances[uuid] = cls(uuid=uuid)
        return cls._instances[uuid]



@dataclass
class Biosample:
    """
    Dataclass to hold parsed information for biosample
    """
    uuid: str
    collection_date : str = field(init=False)
    tissue : list[str] = field(init=False)
    donors : list[Donor] = field(init=False)

    _instances: ClassVar[dict[str, "Biosample"]] = {}  # Class-level registry

    def __post_init__(self):
        self.json_object : object = lattice.get_object(self.uuid,conn,frame='object')
        self.collection_date : str = self.json_object['date_obtained'] if 'date_obtained' in self.json_object else None #date_obtained
        self.tissue : list[str] = [lattice.get_object((f.split('/')[2]),conn,frame='object')['term_name'] for f in self.json_object['sample_terms']]      #sample_terms (this information is also available in the library json object if we wanna just grab it there instead?)
        #self.age : str = #combine lower and upper bound age range and age units to calculate development stage?
        self.donors : list[Donor] = [Donor.get_or_create(f.split('/')[2]) for f in self.json_object['donors']]
    
    @classmethod
    def get_or_create(cls, uuid: str) -> "Biosample":
        """Return existing Biosample if UUID exists, otherwise create and register a new one."""
        if uuid not in cls._instances:
            cls._instances[uuid] = cls(uuid=uuid)
        return cls._instances[uuid]

@dataclass
class Library:
    """
    Dataclass to hold parsed information for library
    """
    uuid: str
    library_id : str = field(init=False)
    samples : list[Biosample] = field(init=False)
    library_strategy : str = field(init=False)
    design_description : str = field(init=False)
    _instances: ClassVar[dict[str, "Library"]] = {}

    def __post_init__(self):
        self.json_object : object = lattice.get_object(self.uuid,conn,frame='object')
        self.library_id : str = self.json_object['aliases'][0].split(":")[1] # Add check that list is length 1? This is the issue with aliases being unstructured
        self.samples: list[Biosample] = [Biosample.get_or_create(f.split('/')[2]) for f in self.json_object['samples']]
        self.library_strategy : str = self.json_object['chemistry_version'] if 'chemistry_version' in self.json_object else None
        self.design_description : str = self.json_object['description'] if 'description' in self.json_object else None

    @classmethod
    def get_or_create(cls, uuid: str) -> "Library":
        if uuid not in cls._instances:
            cls._instances[uuid] = cls(uuid=uuid)
        return cls._instances[uuid]

    @property
    def tissues(self) -> list[str]:
        return sorted(set(t for b in self.samples for t in b.tissue))

    @property
    def collection_dates(self) -> list[str]:
        dates = sorted(set(b.collection_date for b in self.samples if b.collection_date is not None))
        return dates if dates else None
        
    @property
    def donors(self) -> list[Donor]:
        seen = set()
        return [d for b in self.samples for d in b.donors if not (d.uuid in seen or seen.add(d.uuid))]

    @property
    def donor_sexes(self) -> list[str]:
        return sorted(set(d.sex for b in self.samples for d in b.donors))

    @property
    def donor_ethnicities(self) -> list[str]:
        return sorted(set(d.ethnicity for b in self.samples for d in b.donors))

    @classmethod
    def get_or_create(cls, uuid: str) -> "Library":
        if uuid not in cls._instances:
            cls._instances[uuid] = cls(uuid=uuid)
        return cls._instances[uuid]
    

In [14]:
Libraries = [Library.get_or_create(f) for f in library_uuids]

In [15]:
def build_dataframe(libraries: list[Library]) -> pd.DataFrame:
    """
    Flatten all nested Library -> Biosample -> Donor objects into a
    DataFrame with one row per library, indexed by library_id.
    """
    records = []
    for lib in libraries:
        records.append({
            "library_id":         lib.library_id,
            "library_strategy":   lib.library_strategy,
            "design_description": lib.design_description,
            "collection_date":    lib.collection_dates,
            "tissue":             lib.tissues,
            "donor_sex":          lib.donor_sexes,
            "donor_ethnicity":    lib.donor_ethnicities,
        })

    df = pd.DataFrame(records)
    df = df.set_index("library_id")
    return df


df = build_dataframe(Library._instances.values())

In [16]:
check = build_dataframe(Libraries)
check

,library_strategy,design_description,collection_date,tissue,donor_sex,donor_ethnicity
library_id,,,,,,
CD4i_R1L08_CRI,Flex v1,None,None,[blood],[female],"[African American, Hispanic or Latin]"
CD4i_R1L02_CRI,Flex v1,None,None,[blood],[female],"[African American, Hispanic or Latin]"
CD4i_R1L06_CRI,Flex v1,None,None,[blood],[female],"[African American, Hispanic or Latin]"


In [106]:
def calculate_and_assign_dev_stage:
    """
    Takes upper and lower age bounds and age units and returns dev stage term id
    """
    

SyntaxError: unterminated triple-quoted string literal (detected at line 2) (3179511923.py, line 2)

In [ ]:
#Same, but for files, and starting with RawMatrixFile list

In [29]:
@dataclass
class SequenceFile:
    """
    Dataclass to hold parsed information for SequenceFile
    """
    uuid: str
    file_type : str = field(init=False)
    library_id: str = field(init=False)
    
    

    _instances: ClassVar[dict[str, "SequenceFile"]] = {}  # Class-level registry

    def __post_init__(self):
        self.json_object : object = lattice.get_object(self.uuid,conn,frame='object')
        self.file_type : str = self.json_object['file_format']
        self.library_id : str = self.json_object['aliases'][0].split(":")[1]
    
    @classmethod
    def get_or_create(cls, uuid: str) -> "SequenceFile":
        """Return existing SequenceFile if UUID exists, otherwise create and register a new one."""
        if uuid not in cls._instances:
            cls._instances[uuid] = cls(uuid=uuid)
        return cls._instances[uuid]


@dataclass
class RawMatrixFile:
    """
    Dataclass to hold parsed information for RawMatrixFile
    """
    uuid: str
    s3_uri: str = field(init=False)
    sequence_files : list[SequenceFile] = field(init=False)
    #sequence_file_sets : list[SequenceFileSet] = field(init=False)
    _instances: ClassVar[dict[str, "RawMatrixFile"]] = {}

    def __post_init__(self):
        self.json_object : object = lattice.get_object(self.uuid,conn,frame='object')
        self.s3_uri : str = self.json_object['s3_uri']
        self.sequence_files : list[SequenceFile] = [SequenceFile.get_or_create(f) for f in self.json_object['derived_from']]

    @property
    def file_types(self) -> list[str]:
        return sorted(set(f.file_type for f in self.sequence_files))

    @property
    def library_ids(self) -> list[str]:
        return sorted(set(f.library_id for f in self.sequence_files))
    
    @classmethod
    def get_or_create(cls, uuid: str) -> "RawMatrixFile":
        """Return existing RawMatrixFile if UUID exists, otherwise create and register a new one."""
        if uuid not in cls._instances:
            cls._instances[uuid] = cls(uuid=uuid)
        return cls._instances[uuid]

In [30]:
raw_matrix_file_uuids = ['5452309d-17e4-4444-9590-7eea5090721d', 'b1dbb3df-ec51-4902-b1c4-9c17708cddf6', 
                         '7f1efa38-12e1-4511-b1ee-3f7a1bb85ef5', '767b529e-4eb1-442b-913c-79ffdeb841a9']

In [31]:
RawMatrixFiles = [RawMatrixFile.get_or_create(f) for f in raw_matrix_file_uuids]

In [33]:
def build_rawmatrix_dataframe(raw_matrix_files) -> pd.DataFrame:
    """
    Flatten all RawMatrixFile objects into a DataFrame indexed by s3_uri.
    """
    records = []
    for rmf in raw_matrix_files:
        records.append({
            "s3_uri":      rmf.s3_uri,
            "file_types":  rmf.file_types,
            "library_ids": rmf.library_ids,
        })

    df = pd.DataFrame(records)
    df = df.set_index("s3_uri")
    return df


df = build_rawmatrix_dataframe(RawMatrixFile._instances.values())
df

,file_types,library_ids
s3_uri,,
s3://czi-psomagen/marson-mapping-grns-perturb-seq/AN00024294/CD4i_R1L01/processed/cellranger/Run_2025-05-23/outs/per_sample_outs/CD4i_R1L01_D1_Rest/count/sample_filtered_feature_bc_matrix.h5,[fastq],"[418762-CD4i_R1L01-CRI-R1, 418762-CD4i_R1L01-C..."
s3://czi-psomagen/marson-mapping-grns-perturb-seq/AN00024294/CD4i_R1L01/processed/cellranger/Run_2025-05-23/outs/per_sample_outs/CD4i_R1L01_D1_Stim8hr/count/sample_filtered_feature_bc_matrix.h5,[fastq],"[418762-CD4i_R1L01-CRI-R1, 418762-CD4i_R1L01-C..."
s3://czi-psomagen/marson-mapping-grns-perturb-seq/AN00024294/CD4i_R1L01/processed/cellranger/Run_2025-05-23/outs/per_sample_outs/CD4i_R1L01_D2_Rest/count/sample_filtered_feature_bc_matrix.h5,[fastq],"[418762-CD4i_R1L01-CRI-R1, 418762-CD4i_R1L01-C..."
s3://czi-psomagen/marson-mapping-grns-perturb-seq/AN00024294/CD4i_R1L04/processed/cellranger/Run_2025-05-23/outs/per_sample_outs/CD4i_R1L04_D2_Rest/count/sample_filtered_feature_bc_matrix.h5,[fastq],"[418762-CD4i_R1L04-CRI-R1, 418762-CD4i_R1L04-C..."


In [ ]:
# Difficult to incorporate SequenceFileSet, as there isn't a direct relation between SequenceFiles and SequenceFileSets other than sharing the same Library